# NLP Lab 5: N-gram Language Models

Tokenization, N-gram counting, unigram / bigram / trigram models, the zero-probability problem, smoothing and perplexity.

**Conventions (from the task sheet)**
- everything is lowercased, and each sentence gets `<s>` and `</s>` markers
- the vocabulary `V` = all words plus `</s>` (**no** `<s>`)
- natural logarithm (`ln`) everywhere; values are rounded to 4 decimals (very small probabilities are shown in scientific notation with 4 significant digits, since they would otherwise round to 0.0000)
- standard library only (`math`, `itertools`, `collections`)

**Modelling choices**
- `<s>` is only ever a *context*, never a predicted word, so it is not counted in the unigram total `N`.
- `</s>` is never a context (nothing follows it), so it is excluded from the "previous word" loops in the smoothing sums.

In [1]:
import math
import itertools
from collections import Counter

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test  = ["the cat sat on the rug", "the fish sat on the mat"]

def fp(p):
    # format a probability: 4 decimals normally, scientific notation when tiny
    if p == 0:
        return "0"
    if p == math.inf:
        return "inf"
    return f"{p:.4f}" if p >= 1e-3 else f"{p:.4e}"

## Task 1: Tokenization

In [2]:
def tokenize(sentence):
    # lowercase, split on spaces, add sentence markers
    return ["<s>"] + sentence.lower().split() + ["</s>"]

train_tokens = [tokenize(s) for s in train]
test_tokens  = [tokenize(s) for s in test]

print("TRAIN")
for t in train_tokens:
    print(" ", t)
print("TEST")
for t in test_tokens:
    print(" ", t)

with_markers = sum(len(t) for t in train_tokens)
without_markers = sum(len(t) - 2 for t in train_tokens)
print()
print("Total training tokens WITH markers   :", with_markers)
print("Total training tokens WITHOUT markers:", without_markers)

TRAIN
  ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
  ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
  ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
  ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
TEST
  ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
  ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Total training tokens WITH markers   : 30
Total training tokens WITHOUT markers: 22


In [3]:
def build_vocab(sentences):
    # all words plus </s>, but NOT <s>
    vocab = set()
    for s in sentences:
        for tok in tokenize(s):
            if tok != "<s>":
                vocab.add(tok)
    return sorted(vocab)

vocab = build_vocab(train)
V = len(vocab)
print("V =", V)
print("Vocabulary:", vocab)

V = 11
Vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2: Generating N-grams

In [4]:
def get_ngrams(tokens, n):
    # list of n-tuples; returns [] if n is larger than the sentence (or n < 1)
    if n < 1 or n > len(tokens):
        return []
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

first = train_tokens[0]
print("Sentence:", first)
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    print(f"\n{name}:")
    print(get_ngrams(first, n))

Sentence: ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']

Unigrams:
[('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]

Bigrams:
[('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]

Trigrams:
[('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]


In [5]:
# verify: m tokens -> m - n + 1 n-grams, for n = 1, 2, 3, for every sentence
for toks in train_tokens + test_tokens:
    m = len(toks)
    for n in (1, 2, 3):
        assert len(get_ngrams(toks, n)) == m - n + 1, (toks, n)
print("OK: for all", len(train_tokens + test_tokens), "sentences, m tokens give m - n + 1 n-grams for n = 1, 2, 3")

# n larger than the sentence length: no crash, returns an empty list
short = ["<s>", "hi", "</s>"]
print("\nget_ngrams(3 tokens, n=5)  ->", get_ngrams(short, 5))
print("get_ngrams(3 tokens, n=10) ->", get_ngrams(short, 10))
print("get_ngrams([], n=2)        ->", get_ngrams([], 2))

OK: for all 6 sentences, m tokens give m - n + 1 n-grams for n = 1, 2, 3

get_ngrams(3 tokens, n=5)  -> []
get_ngrams(3 tokens, n=10) -> []
get_ngrams([], n=2)        -> []


## Task 3: Counting N-grams

In [6]:
def count_ngrams(token_lists, n):
    c = Counter()
    for toks in token_lists:
        c.update(get_ngrams(toks, n))
    return c

uni_counts = count_ngrams(train_tokens, 1)
bi_counts  = count_ngrams(train_tokens, 2)
tri_counts = count_ngrams(train_tokens, 3)

def sorted_table(counter):
    # highest count first, ties broken alphabetically (deterministic)
    return sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))

def show_table(counter, title):
    print(f"--- {title} ({len(counter)} distinct) ---")
    for gram, c in sorted_table(counter):
        print(f"  {' '.join(gram):<22}{c}")
    print()

show_table(uni_counts, "Unigram counts")
show_table(bi_counts,  "Bigram counts")
show_table(tri_counts, "Trigram counts")

--- Unigram counts (12 distinct) ---
  the                   8
  </s>                  4
  <s>                   4
  ate                   2
  cat                   2
  dog                   2
  on                    2
  sat                   2
  bone                  1
  fish                  1
  mat                   1
  rug                   1

--- Bigram counts (18 distinct) ---
  <s> the               4
  ate the               2
  on the                2
  sat on                2
  the cat               2
  the dog               2
  bone </s>             1
  cat ate               1
  cat sat               1
  dog ate               1
  dog sat               1
  fish </s>             1
  mat </s>              1
  rug </s>              1
  the bone              1
  the fish              1
  the mat               1
  the rug               1

--- Trigram counts (19 distinct) ---
  <s> the cat           2
  <s> the dog           2
  sat on the            2
  ate the bone          1
  at

In [7]:
print("5 most common bigrams:")
for gram, c in sorted_table(bi_counts)[:5]:
    print(f"  {gram}  ->  {c}")

print("\nDistinct unigrams:", len(uni_counts))
print("Distinct bigrams :", len(bi_counts))
print("Distinct trigrams:", len(tri_counts))

5 most common bigrams:
  ('<s>', 'the')  ->  4
  ('ate', 'the')  ->  2
  ('on', 'the')  ->  2
  ('sat', 'on')  ->  2
  ('the', 'cat')  ->  2

Distinct unigrams: 12
Distinct bigrams : 18
Distinct trigrams: 19


In [8]:
def count_of(ngram):
    # count of an n-gram (tuple or space-separated string); 0 if never seen
    if isinstance(ngram, str):
        ngram = tuple(ngram.split())
    table = {1: uni_counts, 2: bi_counts, 3: tri_counts}.get(len(ngram))
    if table is None:
        return 0
    return table.get(ngram, 0)

print("seen   bigram ('the', 'cat') :", count_of(("the", "cat")))
print("unseen bigram ('fish', 'sat'):", count_of(("fish", "sat")))

seen   bigram ('the', 'cat') : 2
unseen bigram ('fish', 'sat'): 0


## Task 4: Unigram model

`P(w) = C(w) / N`, where `N` is the number of predicted tokens in the training data (all words plus `</s>`; `<s>` is excluded, so the probabilities over `V` sum to 1).

In [9]:
N = sum(count_of((w,)) for w in vocab)
print("N (training tokens, <s> excluded) =", N)

def unigram_prob(w):
    return count_of((w,)) / N

print(f"\n{'word':<8}{'C(w)':>6}{'P(w)':>10}")
for w in vocab:
    print(f"{w:<8}{count_of((w,)):>6}{unigram_prob(w):>10.4f}")

total = sum(unigram_prob(w) for w in vocab)
assert math.isclose(total, 1.0)
print("\nSum of probabilities =", round(total, 4), "  (assert passed)")

N (training tokens, <s> excluded) = 26

word      C(w)      P(w)
</s>         4    0.1538
ate          2    0.0769
bone         1    0.0385
cat          2    0.0769
dog          2    0.0769
fish         1    0.0385
mat          1    0.0385
on           2    0.0769
rug          1    0.0385
sat          2    0.0769
the          8    0.3077

Sum of probabilities = 1.0   (assert passed)


In [10]:
def unigram_sentence_prob(sentence):
    # product of P(w) over every word and </s>  (<s> is not predicted)
    p = 1.0
    for w in tokenize(sentence)[1:]:
        p *= unigram_prob(w)
    return p

for s in test:
    p = unigram_sentence_prob(s)
    print(f"{s!r}")
    print(f"   P = {fp(p)}    ln P = {math.log(p):.4f}")

'the cat sat on the rug'
   P = 2.5499e-07    ln P = -15.1821
'the fish sat on the mat'
   P = 1.2749e-07    ln P = -15.8752


In [11]:
# Different word orders -> same unigram probability (unigram ignores order)
words = test[0].split()
perms = sorted(set(itertools.permutations(words)))
probs = {perm: unigram_sentence_prob(" ".join(perm)) for perm in perms}

distinct_values = set(round(p, 15) for p in probs.values())
print("Words:", words)
print("Distinct word orders (permutations):", len(perms))
print("Distinct unigram probability values :", len(distinct_values))
print("Original order probability          :", fp(unigram_sentence_prob(test[0])))
print()
print("A few orders and their probabilities:")
for perm in perms[:2] + perms[len(perms)//2: len(perms)//2 + 2] + perms[-2:]:
    print(f"  {' '.join(perm):<26} P = {fp(probs[perm])}")

assert len(distinct_values) == 1
print("\nAll orders give the same probability, so the unigram model cannot tell")
print("a grammatical sentence from a scrambled one.")

Words: ['the', 'cat', 'sat', 'on', 'the', 'rug']
Distinct word orders (permutations): 360
Distinct unigram probability values : 1
Original order probability          : 2.5499e-07

A few orders and their probabilities:
  cat on rug sat the the     P = 2.5499e-07
  cat on rug the sat the     P = 2.5499e-07
  sat cat on rug the the     P = 2.5499e-07
  sat cat on the rug the     P = 2.5499e-07
  the the sat rug cat on     P = 2.5499e-07
  the the sat rug on cat     P = 2.5499e-07

All orders give the same probability, so the unigram model cannot tell
a grammatical sentence from a scrambled one.


## Task 5: Bigram model (MLE)

`P(w | v) = C(v, w) / C(v)`

In [12]:
def bigram_prob(prev, w):
    c_prev = count_of((prev,))
    if c_prev == 0:
        return 0.0
    return count_of((prev, w)) / c_prev

def next_word_distribution(prev):
    dist = {}
    for (v, w), c in bi_counts.items():
        if v == prev:
            dist[w] = bigram_prob(prev, w)
    return dict(sorted(dist.items()))

dist_the = next_word_distribution("the")
print("P(next | 'the') =")
for w, p in dist_the.items():
    print(f"  {w:<6}{p:.4f}")
assert math.isclose(sum(dist_the.values()), 1.0)
print("sum =", round(sum(dist_the.values()), 4), "  (assert passed)")

P(next | 'the') =
  bone  0.1250
  cat   0.2500
  dog   0.2500
  fish  0.1250
  mat   0.1250
  rug   0.1250
sum = 1.0   (assert passed)


In [13]:
def bigram_sentence_prob(sentence):
    # chain rule: product of P(w_i | w_{i-1}) including <s> -> w1 and w_n -> </s>
    toks = tokenize(sentence)
    p = 1.0
    for prev, w in get_ngrams(toks, 2):
        p *= bigram_prob(prev, w)
    return p

for s in test:
    p = bigram_sentence_prob(s)
    print(f"{s!r}")
    print(f"   P = {fp(p)}")

print("\nScrambled order:", repr("rug the sat the on cat"), "->", bigram_sentence_prob("rug the sat the on cat"))

'the cat sat on the rug'
   P = 0.0156
'the fish sat on the mat'
   P = 0

Scrambled order: 'rug the sat the on cat' -> 0.0


In [14]:
def predict_next(prev):
    # most probable next word; ties broken alphabetically; None if prev never has a successor
    dist = next_word_distribution(prev)
    if not dist:
        return None
    return min(dist, key=lambda w: (-dist[w], w))

for w in ["<s>", "the", "cat", "dog", "sat", "on", "ate", "mat"]:
    dist = next_word_distribution(w)
    best = predict_next(w)
    print(f"predict_next({w!r:7}) = {best!r:7}  P = {dist[best]:.4f}   (candidates: {dist})")

predict_next('<s>'  ) = 'the'    P = 1.0000   (candidates: {'the': 1.0})
predict_next('the'  ) = 'cat'    P = 0.2500   (candidates: {'bone': 0.125, 'cat': 0.25, 'dog': 0.25, 'fish': 0.125, 'mat': 0.125, 'rug': 0.125})
predict_next('cat'  ) = 'ate'    P = 0.5000   (candidates: {'ate': 0.5, 'sat': 0.5})
predict_next('dog'  ) = 'ate'    P = 0.5000   (candidates: {'ate': 0.5, 'sat': 0.5})
predict_next('sat'  ) = 'on'     P = 1.0000   (candidates: {'on': 1.0})
predict_next('on'   ) = 'the'    P = 1.0000   (candidates: {'the': 1.0})
predict_next('ate'  ) = 'the'    P = 1.0000   (candidates: {'the': 1.0})
predict_next('mat'  ) = '</s>'   P = 1.0000   (candidates: {'</s>': 1.0})


## Task 6: Trigram model

`P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2)`

In [15]:
def trigram_prob(w1, w2, w3):
    c_ctx = count_of((w1, w2))
    if c_ctx == 0:
        return 0.0
    return count_of((w1, w2, w3)) / c_ctx

def trigram_sentence_prob(sentence):
    # first word given <s> uses the bigram model, every later word uses the trigram model
    toks = tokenize(sentence)
    p = bigram_prob("<s>", toks[1])
    for w1, w2, w3 in get_ngrams(toks, 3):
        p *= trigram_prob(w1, w2, w3)
    return p

for s in test:
    print(f"{s!r}")
    print(f"   P(trigram model) = {fp(trigram_sentence_prob(s))}")

'the cat sat on the rug'
   P(trigram model) = 0.1250
'the fish sat on the mat'
   P(trigram model) = 0


In [16]:
print(f"{'N-gram type':<14}{'Distinct':>10}")
print(f"{'unigrams':<14}{len(uni_counts):>10}")
print(f"{'bigrams':<14}{len(bi_counts):>10}")
print(f"{'trigrams':<14}{len(tri_counts):>10}")

N-gram type     Distinct
unigrams              12
bigrams               18
trigrams              19


In [17]:
def unseen_ngrams(sentence, n):
    return [g for g in get_ngrams(tokenize(sentence), n) if count_of(g) == 0]

total_unseen = {2: 0, 3: 0}
for s in test:
    ub = unseen_ngrams(s, 2)
    ut = unseen_ngrams(s, 3)
    total_unseen[2] += len(ub)
    total_unseen[3] += len(ut)
    print(f"{s!r}")
    print(f"   unseen bigrams  ({len(ub)}): {ub}")
    print(f"   unseen trigrams ({len(ut)}): {ut}")
    if len(ut) > len(ub):
        winner = "trigram"
    elif len(ub) > len(ut):
        winner = "bigram"
    else:
        winner = "neither (tie)"
    print(f"   -> more unseen N-grams: {winner} model")

print(f"\nTotal over both test sentences: {total_unseen[2]} unseen bigrams vs {total_unseen[3]} unseen trigrams")
print("Higher-order models are sparser: the longer the context, the more N-grams are unseen.")

'the cat sat on the rug'
   unseen bigrams  (0): []
   unseen trigrams (0): []
   -> more unseen N-grams: neither (tie) model
'the fish sat on the mat'
   unseen bigrams  (1): [('fish', 'sat')]
   unseen trigrams (3): [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
   -> more unseen N-grams: trigram model

Total over both test sentences: 1 unseen bigrams vs 3 unseen trigrams
Higher-order models are sparser: the longer the context, the more N-grams are unseen.


## Task 7: Zero-probability problem

In [18]:
def unseen_bigrams(sentence):
    return [bg for bg in get_ngrams(tokenize(sentence), 2) if count_of(bg) == 0]

for s in test:
    print(f"{s!r}")
    print("   unseen bigrams  :", unseen_bigrams(s))
    print("   bigram MLE prob :", fp(bigram_sentence_prob(s)))

assert bigram_sentence_prob(test[0]) > 0
assert bigram_sentence_prob(test[1]) == 0
print("\nConfirmed: the second test sentence has bigram MLE probability exactly 0.")

'the cat sat on the rug'
   unseen bigrams  : []
   bigram MLE prob : 0.0156
'the fish sat on the mat'
   unseen bigrams  : [('fish', 'sat')]
   bigram MLE prob : 0

Confirmed: the second test sentence has bigram MLE probability exactly 0.


In [19]:
# Possible word pairs: words only (markers excluded) -> |words| x |words| ordered pairs
words_only = [w for w in vocab if w != "</s>"]
possible = len(words_only) ** 2
seen_pairs = {bg for bg in bi_counts if bg[0] in words_only and bg[1] in words_only}
never = possible - len(seen_pairs)

print("Word types (markers excluded):", len(words_only))
print("Possible ordered word pairs  :", possible)
print("Distinct pairs seen in train :", len(seen_pairs))
print("Pairs never seen             :", never)
print(f"Percentage never seen        : {100 * never / possible:.1f}%")

Word types (markers excluded): 10
Possible ordered word pairs  : 100
Distinct pairs seen in train : 13
Pairs never seen             : 87
Percentage never seen        : 87.0%


In [20]:
def safe_log(p):
    # natural log, -inf for p == 0 (no math domain error)
    return -math.inf if p == 0 else math.log(p)

# log probability of the second test sentence, built from the individual bigram probabilities
toks = tokenize(test[1])
log_p = 0.0
for prev, w in get_ngrams(toks, 2):
    p = bigram_prob(prev, w)
    print(f"  P({w:<5}| {prev:<4}) = {p:.4f}   ln = {safe_log(p)}")
    log_p += safe_log(p)

print("\nln P(test[1]) =", log_p)
print("safe_log(0)   =", safe_log(0))

  P(the  | <s> ) = 1.0000   ln = 0.0
  P(fish | the ) = 0.1250   ln = -2.0794415416798357
  P(sat  | fish) = 0.0000   ln = -inf
  P(on   | sat ) = 1.0000   ln = 0.0
  P(the  | on  ) = 1.0000   ln = 0.0
  P(mat  | the ) = 0.1250   ln = -2.0794415416798357
  P(</s> | mat ) = 1.0000   ln = 0.0

ln P(test[1]) = -inf
safe_log(0)   = -inf


## Task 8: Smoothing

Add-one (Laplace): `P(w | v) = (C(v, w) + 1) / (C(v) + V)`.

In [21]:
def bigram_prob_addone(prev, w):
    return (count_of((prev, w)) + 1) / (count_of((prev,)) + V)

print(f"{'probability':<16}{'MLE':>10}{'Add-one':>10}")
for prev, w in [("fish", "sat"), ("the", "cat")]:
    print(f"P({w} | {prev})".ljust(16) + f"{bigram_prob(prev, w):>10.4f}{bigram_prob_addone(prev, w):>10.4f}")

probability            MLE   Add-one
P(sat | fish)       0.0000    0.0833
P(cat | the)        0.2500    0.1579


In [22]:
# every possible previous word: <s> and all words (</s> is never a context)
prev_words = ["<s>"] + [w for w in vocab if w != "</s>"]

for prev in prev_words:
    total = sum(bigram_prob_addone(prev, w) for w in vocab)
    assert math.isclose(total, 1.0), (prev, total)
    print(f"  prev = {prev:<5} sum over V = {total:.4f}")
print(f"\nAll {len(prev_words)} smoothed distributions sum to 1  (asserts passed)")

  prev = <s>   sum over V = 1.0000
  prev = ate   sum over V = 1.0000
  prev = bone  sum over V = 1.0000
  prev = cat   sum over V = 1.0000
  prev = dog   sum over V = 1.0000
  prev = fish  sum over V = 1.0000
  prev = mat   sum over V = 1.0000
  prev = on    sum over V = 1.0000
  prev = rug   sum over V = 1.0000
  prev = sat   sum over V = 1.0000
  prev = the   sum over V = 1.0000

All 11 smoothed distributions sum to 1  (asserts passed)


In [23]:
def bigram_prob_addk(prev, w, k, bi=None, uni=None):
    # (C(v,w) + k) / (C(v) + k*V); optional count tables allow training on a subset of the data
    bi  = bi_counts  if bi  is None else bi
    uni = uni_counts if uni is None else uni
    return (bi.get((prev, w), 0) + k) / (uni.get((prev,), 0) + k * V)

ks = [1, 0.5, 0.1, 0.01]
print(f"{'k':<8}{'P(sat | fish)':>15}")
for k in ks:
    print(f"{k:<8}{bigram_prob_addk('fish', 'sat', k):>15.4f}")
print("\nSmaller k moves less probability mass to unseen bigrams.")

k         P(sat | fish)
1                0.0833
0.5              0.0769
0.1              0.0476
0.01             0.0090

Smaller k moves less probability mass to unseen bigrams.


## Task 9: Perplexity and model comparison

`PP = exp( -(1/T) * sum ln P(sentence) )`, where `T` is the total number of predicted tokens (words + `</s>`) in the evaluation sentences.

In [24]:
def perplexity(sentences, prob_fn):
    # prob_fn(sentence) -> probability of that sentence; returns inf if any probability is 0
    log_sum = 0.0
    T = 0
    for s in sentences:
        p = prob_fn(s)
        if p == 0:
            return math.inf
        log_sum += math.log(p)
        T += len(tokenize(s)) - 1          # predicted tokens (no <s>)
    return math.exp(-log_sum / T)

def bigram_sentence_prob_with(sentence, prob_bigram):
    p = 1.0
    for prev, w in get_ngrams(tokenize(sentence), 2):
        p *= prob_bigram(prev, w)
    return p

def unigram_addone_prob(w):
    return (count_of((w,)) + 1) / (N + V)

def unigram_addone_sentence_prob(sentence):
    p = 1.0
    for w in tokenize(sentence)[1:]:
        p *= unigram_addone_prob(w)
    return p

results = [
    ("Bigram MLE",     perplexity(test, bigram_sentence_prob)),
    ("Bigram add-one", perplexity(test, lambda s: bigram_sentence_prob_with(s, bigram_prob_addone))),
    ("Unigram add-one", perplexity(test, unigram_addone_sentence_prob)),
]

print(f"{'Model':<18}{'Perplexity':>12}")
for name, pp in results:
    print(f"{name:<18}{pp:>12.4f}" if pp != math.inf else f"{name:<18}{'inf':>12}")

Model               Perplexity
Bigram MLE                 inf
Bigram add-one          5.7969
Unigram add-one         9.1370


In [25]:
addk_results = {}
for k in ks:
    addk_results[k] = perplexity(test, lambda s, k=k: bigram_sentence_prob_with(s, lambda a, b: bigram_prob_addk(a, b, k)))

print(f"{'k':<8}{'Perplexity':>12}")
for k, pp in addk_results.items():
    print(f"{k:<8}{pp:>12.4f}")

best_k_test_full = min(addk_results, key=addk_results.get)
print(f"\nBest k (bigram add-k, trained on all 4 sentences, evaluated on test) = {best_k_test_full}"
      f"  (perplexity {addk_results[best_k_test_full]:.4f})")

k         Perplexity
1             5.7969
0.5           4.5703
0.1           2.9559
0.01          2.6272

Best k (bigram add-k, trained on all 4 sentences, evaluated on test) = 0.01  (perplexity 2.6272)


In [26]:
# Train on the first 3 sentences only (V is still built from all four), validate on the 4th
train3_tokens = train_tokens[:3]
val = train[3:]
uni3 = count_ngrams(train3_tokens, 1)
bi3  = count_ngrams(train3_tokens, 2)
print("Training on :", train[:3])
print("Validation  :", val)
print("V (from all 4 sentences) =", V)

def pp_addk(sentences, k):
    return perplexity(sentences, lambda s: bigram_sentence_prob_with(s, lambda a, b: bigram_prob_addk(a, b, k, bi3, uni3)))

val_pp  = {k: pp_addk(val, k)  for k in ks}
test_pp = {k: pp_addk(test, k) for k in ks}

print(f"\n{'k':<8}{'Validation PP':>15}{'Test PP':>12}")
for k in ks:
    print(f"{k:<8}{val_pp[k]:>15.4f}{test_pp[k]:>12.4f}")

best_k_val  = min(val_pp,  key=val_pp.get)
best_k_test = min(test_pp, key=test_pp.get)

print(f"\nBest k chosen on validation data : {best_k_val}")
print(f"Test perplexity with that k      : {test_pp[best_k_val]:.4f}")
print(f"Best k found directly on test set: {best_k_test}  (test perplexity {test_pp[best_k_test]:.4f})")

if best_k_val == best_k_test:
    print("\nThe validation set picked the same k as the test set, so there is no loss from tuning on held-out data.")
else:
    gap = test_pp[best_k_val] - test_pp[best_k_test]
    print(f"\nThe validation k differs from the test-optimal k; the test perplexity is {gap:.4f} higher.")
    print("Tuning directly on the test set gives an optimistic estimate, so the validation-based number is the honest one.")

Training on : ['the cat sat on the mat', 'the cat ate the fish', 'the dog sat on the rug']
Validation  : ['the dog ate the bone']
V (from all 4 sentences) = 11

k         Validation PP     Test PP
1                8.5857      5.7406
0.5              8.0163      4.4539
0.1              8.0260      2.7778
0.01            13.1304      2.4258

Best k chosen on validation data : 0.5
Test perplexity with that k      : 4.4539
Best k found directly on test set: 0.01  (test perplexity 2.4258)

The validation k differs from the test-optimal k; the test perplexity is 2.0280 higher.
Tuning directly on the test set gives an optimistic estimate, so the validation-based number is the honest one.


### Summary

- **Unigram** models ignore word order, so any permutation of a sentence gets the same score.
- **Bigram/trigram MLE** capture order but give probability 0 to any unseen N-gram, which makes the sentence probability (and perplexity) zero / infinite. Trigrams are sparser than bigrams.
- **Smoothing** (add-one, add-k) reserves probability mass for unseen N-grams, so perplexity becomes finite. Add-one takes a lot of mass from seen events on a tiny corpus; smaller `k` usually behaves better.
- **`k` should be tuned on validation data**, not on the test set, to avoid an over-optimistic estimate.